# Realtime anomaly detection with ML.NET
**EN** — Monitor cold-chain truck temperatures. Compare ML.NET's IID spike detector with a z-score baseline, run SR-CNN over a whole window, and forecast with SSA.

**ID** — Memantau suhu truk rantai dingin. Bandingkan detektor spike IID ML.NET dengan baseline z-score, jalankan SR-CNN pada satu jendela, dan prakirakan dengan SSA.

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

In [ ]:
// Using a local build instead of nuget.org? Uncomment and point to <repo>/artifacts/nuget (absolute path):
// #i "nuget: C:/src/bigpipe/artifacts/nuget"
#r "nuget: BigPipe.Analytics.ML, 0.1.0"
using BigPipe.Client;
using BigPipe.Client.Admin;
using BigPipe.Analytics;
var bootstrap = Environment.GetEnvironmentVariable("BIGPIPE_BOOTSTRAP") ?? "localhost:9092";
var admin = new BigPipeAdminClient(Environment.GetEnvironmentVariable("BIGPIPE_ADMIN") ?? "http://localhost:9644");
var http = new BigPipeHttpClient(Environment.GetEnvironmentVariable("BIGPIPE_HTTP") ?? "http://localhost:8082");
string Unique(string p) => $"nb-{p}-{Guid.NewGuid().ToString("N")[..6]}";
var cluster = await admin.GetClusterAsync();
Console.WriteLine($"Connected to {cluster.ClusterId} (BigPipe {cluster.Version}) — {cluster.Credit}");

In [ ]:
using BigPipe.Analytics.ML;
var topic = Unique("telemetry");
await admin.CreateTopicAsync(topic, 1, StorageMode.Diskless);   // telemetry is cheap on object storage
var rng = new Random(7);
var spikes = new HashSet<int> { 180, 300, 420 };
var readings = Enumerable.Range(0, 500).Select(i => 5 + Math.Sin(i / 24.0) + rng.NextDouble() * 0.4 + (spikes.Contains(i) ? 8 : 0)).ToList();
await http.ProduceAsync(topic, readings.Select((t, i) => new HttpProduceRecord { Key = "truck-07", Value = new { temp_c = Math.Round(t, 3), i } }));
Console.WriteLine($"produced {readings.Count} readings with faults at {string.Join(", ", spikes)}");

## Streaming detectors / Detektor streaming

In [ ]:
var ml = new MLSpikeDetector(confidence: 99, pvalueHistoryLength: 60);
var z = new ZScoreDetector(threshold: 4);
var alerts = new List<object>();
var i = 0;
await foreach (var r in BigPipeSource.Kafka(bootstrap, topic, from: OffsetReset.Earliest))
{
    var t = r.Num("temp_c");
    var a = ml.Observe(t);
    var b = z.Observe(t);
    if (a.IsAnomaly || b.IsAnomaly) alerts.Add(new { reading = i, temp = t, mlSpike = a.IsAnomaly, zscore = Math.Round(b.Score, 1) });
    if (++i == readings.Count) break;
}
alerts

## SR-CNN over a window and SSA forecast / SR-CNN pada jendela dan prakiraan SSA

In [ ]:
var sr = new SrCnnWindowDetector(threshold: 0.3).Detect(readings);
Console.WriteLine("SR-CNN anomalies at: " + string.Join(", ", sr.Where(p => p.IsAnomaly).Select(p => p.Index)));
var ssa = new SsaForecaster(windowSize: 24, seriesLength: 96, horizon: 6);
ssa.Train(readings.Take(160));
var f = ssa.ObserveAndForecast(readings[160]);
f.ForecastedValues.Select((v, k) => new { step = k + 1, forecast = Math.Round(v, 2), low = Math.Round(f.LowerBound[k], 2), high = Math.Round(f.UpperBound[k], 2) })